In [ ]:
pip install requests beautifulsoup4 pytesseract pillow

In [ ]:
import requests
from bs4 import BeautifulSoup
import pytesseract
from PIL import Image
from io import BytesIO
import re
import pandas as pd
import time

# --- CONFIGURATION ---
BASE_URL = "https://www.topjobs.lk"
# The main listing page
LIST_URL = "https://www.topjobs.lk/applicant/vacancybyfunctionalarea.jsp?FA=SDQ&jst=OPEN"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

def get_job_links():
    print(f"Fetching list from: {LIST_URL}")
    response = requests.get(LIST_URL, headers=HEADERS)
    soup = BeautifulSoup(response.text, 'html.parser')

    job_data = []


    job_code_spans = soup.find_all('span', id=re.compile(r'^hdnJC'))

    print(f"Found {len(job_code_spans)} job rows based on hidden inputs.")

    for jc_span in job_code_spans:
        try:

            jc = jc_span.get_text(strip=True)
            row = jc_span.find_parent('tr')

            if not row:
                continue


            ec_span = row.find('span', id=re.compile(r'^hdnEC'))
            ac_span = row.find('span', id=re.compile(r'^hdnAC'))

            ec = ec_span.get_text(strip=True) if ec_span else "DEFZZZ"
            ac = ac_span.get_text(strip=True) if ac_span else "DEFZZZ"


            title_tag = row.find('h2')
            position = title_tag.get_text(strip=True) if title_tag else "Unknown Position"

            detail_url = f"{BASE_URL}/employer/JobAdvertismentServlet?jc={jc}&ac={ac}&ec={ec}&pg=applicant/vacancybyfunctionalarea.jsp"

            job_data.append({
                "Position": position,
                "URL": detail_url
            })

        except Exception as e:
            print(f"Error parsing row: {e}")
            continue

    return job_data

def scrape_job_description_image(url):
    """
    Downloads the ad image from the detail page and performs OCR.
    """
    try:
        response = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(response.text, 'html.parser')

        images = soup.find_all('img')
        target_img_url = None

        for img in images:
            src = img.get('src', '')

            if any(x in src for x in ['/logo/', '/uploads/', 'defzzz']) and not any(x in src for x in ['small', 'icon', 'button']):
                target_img_url = src if src.startswith('http') else BASE_URL + src
                break

        if not target_img_url:
            return "[No suitable image found]"


        img_response = requests.get(target_img_url, headers=HEADERS)
        img = Image.open(BytesIO(img_response.content))

        # OCR
        text = pytesseract.image_to_string(img)

        # Cleanup
        clean_text = "\n".join([line.strip() for line in text.split('\n') if line.strip()])
        return clean_text

    except Exception as e:
        return f"[Error: {str(e)}]"

# --- EXECUTION ---


jobs = get_job_links()
total_jobs = len(jobs)

if total_jobs == 0:
    print("No jobs found. Check connection or site changes.")
else:
    print(f"Found {total_jobs} jobs. Starting batch processing...")

    results = []
    batch_size = 10  # Save to Csv every 10 jobs
    output_file = "topjobs_all_data.csv"


    for i, job in enumerate(jobs):
        print(f"[{i+1}/{total_jobs}] Scraping: {job['Position']}...")

        # Scrape description
        desc = scrape_job_description_image(job['URL'])

        results.append({
            "Job role(Position)": job['Position'],
            "Job description": desc
        })


        if (i + 1) % batch_size == 0:
            df = pd.DataFrame(results)
            df.to_csv(output_file, index=False)
            print(f"   >>> Progress saved to {output_file} ({i+1} jobs so far)")

        # bprevent IP blocking
        time.sleep(1)

    # Final Save
    df = pd.DataFrame(results)
    df.to_csv(output_file, index=False)
    print(f"\nDONE! All {total_jobs} jobs saved to '{output_file}'.")

Fetching list from: https://www.topjobs.lk/applicant/vacancybyfunctionalarea.jsp?FA=SDQ&jst=OPEN
Found 225 job rows based on hidden inputs.
Found 225 jobs. Starting batch processing...
[1/225] Scraping: System Analyst (Associate/ Trainee)...
[2/225] Scraping: Senior Risk & Fraud Platform Engineer...
[3/225] Scraping: Software Engineer | React Full Stack Web Developer...
[4/225] Scraping: Senior Software Engineers (React.JS)...
[5/225] Scraping: System Implementation Engineer...
[6/225] Scraping: Full Stack Engineer (Mid - Senior Level)...
[7/225] Scraping: Intern - Android Development...
[8/225] Scraping: Intern - Fullstack Developer...
[9/225] Scraping: Graphic Designers...
[10/225] Scraping: IT Support Engineer...
   >>> Progress saved to topjobs_all_data.csv (10 jobs so far)
[11/225] Scraping: IT Support Executive (Female)...
[12/225] Scraping: Graphic Designer | Social Media Manager - Nawala...
[13/225] Scraping: Freelance Quality Assurance (QA) Lead...
[14/225] Scraping: Developer